In [ ]:
!pip install kagglehub

In [1]:
import os, pickle, glob
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml.feature import Word2Vec

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("cynthiarempel/amazon-us-customer-reviews-dataset")

print("Path to dataset files:", path)

Path to dataset files: /home/hkwon2/.cache/kagglehub/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/versions/9


In [4]:
def get_dir_size(path):
    total = 0
    with os.scandir(path) as it:
        for entry in it:
            if entry.is_file():
                total += entry.stat().st_size
            elif entry.is_dir():
                total += get_dir_size(entry.path)
    return total

print(f"File Size: {get_dir_size(path) / (1024**3):.2f} GB")
print("File List:", f"{len(os.listdir(path))} Files" , os.listdir(path))

File Size: 50.68 GB
File List: 37 Files ['amazon_reviews_us_Books_v1_02.tsv', 'amazon_reviews_us_Video_v1_00.tsv', 'amazon_reviews_us_Health_Personal_Care_v1_00.tsv', 'amazon_reviews_us_Sports_v1_00.tsv', 'amazon_reviews_us_Digital_Video_Download_v1_00.tsv', 'amazon_reviews_us_Digital_Ebook_Purchase_v1_01.tsv', 'amazon_reviews_us_Pet_Products_v1_00.tsv', 'amazon_reviews_us_Camera_v1_00.tsv', 'amazon_reviews_us_Electronics_v1_00.tsv', 'amazon_reviews_us_Baby_v1_00.tsv', 'amazon_reviews_us_Gift_Card_v1_00.tsv', 'amazon_reviews_us_Software_v1_00.tsv', 'amazon_reviews_us_Musical_Instruments_v1_00.tsv', 'amazon_reviews_us_Video_DVD_v1_00.tsv', 'amazon_reviews_us_Outdoors_v1_00.tsv', 'amazon_reviews_us_Digital_Video_Games_v1_00.tsv', 'amazon_reviews_us_Wireless_v1_00.tsv', 'amazon_reviews_us_Tools_v1_00.tsv', 'amazon_reviews_us_Mobile_Electronics_v1_00.tsv', 'amazon_reviews_us_Digital_Music_Purchase_v1_00.tsv', 'amazon_reviews_us_Digital_Software_v1_00.tsv', 'amazon_reviews_us_Major_Applianc

In [5]:
df = pd.read_csv(path+'/amazon_reviews_us_Books_v1_02.tsv', sep='\t', nrows=100)

In [6]:
df.head()

,marketplace,customer_id,review_id,product_id,product_parent,product_title,product_category,star_rating,helpful_votes,total_votes,vine,verified_purchase,review_headline,review_body,review_date
0,US,12076615,RQ58W7SMO911M,0385730586,122662979,Sisterhood of the Traveling Pants (Book 1),Books,4,2,3,N,N,this book was a great learning novel!,this boook was a great one that you could lear...,2005-10-14
1,US,12703090,RF6IUKMGL8SF,0811828964,56191234,The Bad Girl's Guide to Getting What You Want,Books,3,5,5,N,N,Fun Fluff,If you are looking for something to stimulate ...,2005-10-14
2,US,12257412,R1DOSHH6AI622S,1844161560,253182049,"Eisenhorn (A Warhammer 40,000 Omnibus)",Books,4,1,22,N,N,this isn't a review,never read it-a young relative idicated he lik...,2005-10-14
3,US,50732546,RATOTLA3OF70O,0373836635,348672532,Colby Conspiracy (Colby Agency),Books,5,2,2,N,N,fine author on her A-game,Though she is honored to be Chicago Woman of t...,2005-10-14
4,US,51964897,R1TNWRKIVHVYOV,0262181533,598678717,The Psychology of Proof: Deductive Reasoning i...,Books,4,0,2,N,N,Execellent cursor examination,Review based on a cursory examination by Unive...,2005-10-14
